# Do Chicago speed cameras reduce crashes?
### Upgraded research notebook

This notebook is the **upgraded** analysis. It reads artefacts produced by the tested
`chicago_cameras` package; it does not replace or modify the original notebook, which remains at the
repository root (`Do_Chicago_speed_cameras_reduce_crashes_.ipynb`) as a historical record.

**What is different from the original:** metric (UTM) distances, explicit local matching, removal of
overlapping camera catchments, a coverage guard for missing early months, a staggered-adoption-robust
estimator, formal pre-trend testing, fit-gated synthetic control, severity outcomes, placebo tests and
a full sensitivity table.

Run the pipeline first:

```bash
python scripts/download_data.py
python -m chicago_cameras.pipeline --config config/analysis.yaml
```


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

estimates_path = RESULTS / "estimates.json"
if not estimates_path.exists():
    raise FileNotFoundError(
        "No results found. Run: python -m chicago_cameras.pipeline --config config/analysis.yaml"
    )
results = json.loads(estimates_path.read_text())
print("Results generated:", results["generated_at_utc"])
print("Analysis window:", results["sample"]["first_month"], "to", results["sample"]["last_month"])


## 1. Sample and design

The estimation sample is deliberately smaller than the set of all cameras. A camera enters only if
it has a complete pre/post window, and only if its 250 m catchment does not overlap a camera that is
already live. Without the second condition, the same crash can be counted by several "independent"
experiments.


In [ ]:
sample = pd.DataFrame([results["sample"]]).T.rename(columns={0: "value"})
display(sample)

diagnostics = results["diagnostics"]
rows = [
    ("Cameras in source", diagnostics["n_cameras_total"]),
    ("Cameras in event window", diagnostics["n_cameras_in_window"]),
    ("Dropped for catchment overlap", diagnostics["n_cameras_dropped_overlap"]),
    ("Matched cameras", diagnostics["n_cameras_matched"]),
    ("Candidate comparison sites", diagnostics["n_control_candidates"]),
    ("Control-to-camera minimum distance (m)", round(diagnostics["control_min_distance_to_camera_m"], 1)),
    ("Treated/control catchment overlap", diagnostics["treated_control_catchment_overlap"]),
    ("Comparison-site overlap pairs", diagnostics["control_overlap_pairs"]),
]
display(pd.DataFrame(rows, columns=["Diagnostic", "Value"]))
print("Panel validation:", results["validation"])


## 2. Main estimates

`pct` is the percent change in crashes for count models. Level effects are in crashes per
site-month. The preferred specifications are the conditional Poisson FE and the per-camera 2x2
average; the original OLS/Poisson specification is retained as a baseline, and the replication rows
use the original sample and citywide matching.


In [ ]:
main = pd.DataFrame(results["estimates"])
columns = ["method", "estimate", "ci_low", "ci_high", "pvalue", "pct",
           "pct_ci_low", "pct_ci_high", "n_clusters", "n_obs"]
display(main[[c for c in columns if c in main.columns]].round(3))

print("Wild cluster bootstrap p-value:",
      round(results["wild_cluster_bootstrap"]["pvalue"], 3))

primary = main[(main["method"].str.contains("Conditional Poisson")) &
               (~main["method"].str.contains("Replication"))].iloc[0]
print(f"Primary estimate: {primary['pct']:+.1f}% "
      f"(95% CI {primary['pct_ci_low']:+.1f}% to {primary['pct_ci_high']:+.1f}%), "
      f"p={primary['pvalue']:.3f}")


In [ ]:
display(Image(filename=str(FIGURES / "04_main_estimates.png")))


## 3. Event study and pre-trends

The joint pre-trend test can fail to reject parallel trends; it cannot prove them, and with about
twenty cameras it has low power.


In [ ]:
event = pd.read_csv(TABLES / "event_study.csv")
display(event.round(3))
print("Joint pre-trend test:", results["event_study_pre_trend"])
display(Image(filename=str(FIGURES / "05_event_study.png")))


## 4. Synthetic control

The synthetic control is reported as a **disagreement**, not as confirmation. Its pre-period fit is
moderate, and its post-period gap points in the opposite direction to the DiD. A fit gate prevents a
weak synthetic control from being presented as evidence.


In [ ]:
sc = results["synthetic_control"]
sc_summary = {k: sc[k] for k in
              ["cohort_go_month", "cohort_size", "pre_r2", "pre_rmspe", "pre_mean",
               "post_rmspe", "post_gap_mean", "pct_vs_synth", "rmspe_ratio",
               "p_two_sided_ratio", "p_one_sided_negative", "informative"]}
display(pd.Series(sc_summary, name="value").to_frame())
display(pd.DataFrame(sc["in_time_placebo"]))
display(pd.DataFrame(sc["leave_one_out"]))
display(Image(filename=str(FIGURES / "06_synthetic_control.png")))


## 5. Robustness and falsification

The table below is the complete pre-registered set of design changes. The headline result is the
baseline row, not the most favourable row.


In [ ]:
sensitivity = pd.read_csv(TABLES / "sensitivity.csv")
display(sensitivity[["analysis", "n_cameras", "did_pct", "did_pct_ci_low",
                     "did_pct_ci_high", "did_pvalue", "status"]].round(2))

print("Lattice-phase range:", results["lattice_phase"])
print("Geographic placebos:", results["placebo_geography"])
display(Image(filename=str(FIGURES / "10_sensitivity.png")))
display(Image(filename=str(FIGURES / "07_placebo_distribution.png")))


## 6. Severity, subgroups and mechanism

Severity outcomes use the same design. Subgroup estimates are exploratory and BH-adjusted. The
violations series measures enforcement output, not speeding behaviour.


In [ ]:
display(pd.DataFrame(results["severity"]).round(2))
display(pd.DataFrame(results["subgroups"]).round(3))
display(Image(filename=str(FIGURES / "09_severity.png")))
display(Image(filename=str(FIGURES / "08_subgroups.png")))

mechanism = pd.DataFrame([results["mechanism"]]).T.rename(columns={0: "value"})
display(mechanism)


## 7. Optional: refit the pipeline

Set `RUN_PIPELINE = True` to refit everything. This takes 15-20 minutes and requires the raw data
(`python scripts/download_data.py`). The dashboard and this notebook otherwise read only cached,
validated artefacts.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "chicago_cameras.pipeline", "--config", "config/analysis.yaml"],
        cwd=ROOT, check=True,
    )


## 8. Where to read more

- `docs/AUDIT.md` — what the original did, what was wrong, and what changed.
- `docs/METHODOLOGY.md` — estimand, assumptions, estimators, interpretation rules.
- `docs/DATA_DICTIONARY.md` — every field, exclusion and provenance record.
- `docs/REPORT.md` — full research report.
- `docs/REPRODUCIBILITY.md` — exact commands and expected outputs.
